# 보존도 분석 대상 세트 — 클러스터 기반 vs identity 기반

ChCODH2(Q9F8A8, 636 aa) 를 inverse folding 으로 재설계하려면 **어느 잔기를 건드려도 되는지**
를 먼저 알아야 하고, 그것은 근연 상동체의 보존도로 판단한다. 그 앞 단계가 이 노트북이다 —
**보존도를 계산할 서열 세트를 어떻게 고를 것인가.**

후보는 두 가지이고, **어느 쪽이 낫다고 미리 정하지 않는다.** 같은 잣대로 재서 숫자만 내놓는다.

| | 방식 | 세트의 정의 | 성질 |
|---|---|---|---|
| **A** | 클러스터 기반 | ChCODH2 가 속한 mmseqs 클러스터의 멤버 | 서열 간 상호 유사도로 묶인 덩어리. 경계가 임계값에 달려 있다 |
| **B** | identity 기반 | ChCODH2 와의 `fident` 가 기준 이상인 서열 전부 | ChCODH2 중심의 방사형. 서로는 멀어도 된다 |

## 01 번 묶음에서 이미 확인된 사실 (추측이 아니라 서버 실행 결과)

`20261025_01_cluster_sweep` 가 IPR010047 1,827 서열을 `-c 0.8 --cov-mode 1 -s 7.5` 로
0.10~0.90 스윕한 결과다. 이 노트북은 **그 산출물을 읽기만 한다.**

| min-seq-id | 0.10 | 0.30 | 0.40 | 0.50 | 0.60 | 0.65 | 0.70 | 0.80 | 0.90 |
|---|---|---|---|---|---|---|---|---|---|
| 클러스터 | 3 | 5 | 11 | 29 | 86 | 140 | 219 | 493 | 921 |
| ChCODH2 그룹 | 1460 | 730 | 381 | 104 | 28 | 19 | 14 | 4 | 4 |

- 2026-04 결과와 **완전히 일치**했다 (master_1827.tsv 와 ARI 1.0000, 1:1 대응 219개). 환경 동일.
- **쓸모있는 해상도(클러스터 11개 이상)에서 ARI 0.90 이상 구간이 하나도 없다.** 최대 0.868
  (0.60→0.65). 이 패밀리는 자연적 경계가 없고 다양성이 연속적이다.
  **0.70 은 "발견된 최적값" 이 아니라 실용적 선택이다.**
- NMI 는 0.55 위에서 전부 0.90 이상 → 분할이 계층적으로 깔끔히 포개진다.
  그래서 0.90 분할을 '중복 제거용 자' 로 쓸 수 있다 (아래 `n_effective`).
- **ChCODH2 의 소속이 0.40~0.55 에서 불안정하다** (0.45→36개, 0.50→104개, 0.55→6개).
  0.60 이상에서야 A0A494X3V7 그룹에 정착한다. → 8절이 이것을 identity 기반에서도 확인한다.
- **ChCODH2 는 전 임계값에서 한 번도 클러스터 대표가 아니다** (`ref_is_mm_rep` 전부 False).
  대표만 쓰는 방식이면 **정작 설계 대상이 빠진다.** → 모든 세트에서 `has_reference` 를 확인한다.
- ChCODH2 대비 identity 중앙값 0.379, 95% 가 0.55 이하, 최대 1.0(자기 자신).

## 이 노트북의 핵심 — `n_effective`

서열 수 `n` 으로 두 방식을 비교하면 **거의 같은 서열 28개가 14개보다 좋아 보인다.** 그렇지 않다.
보존도의 실제 분모는 **서로 다른 서열이 몇 개인가**이므로, 모든 세트를 **0.90 분할 기준
서로 다른 클러스터 수**(`n_effective`)로 다시 센다. `redundancy = 1 - n_effective / n`.
6절 표는 `n` 이 아니라 `n_effective` 로 줄 세운다.

## 전체 흐름

| 절 | 하는 일 | 보고 판단할 것 |
|---|---|---|
| 1 설정 | 경로·후보·기준값 (셀 2개) | 고칠 곳은 `BASE` 와 `RUN` 두 줄 |
| 2 입력 확인 | FASTA·master·alignment 를 센다 | 1,827 인가. alignment 가 절단됐나. **r612 vs 논문 V610** |
| 3 identity 분포 | ChCODH2 대비 fident | 중앙값 0.379 와 맞는가. 각 기준에서 몇 개 남는가 |
| 4 클러스터 세트 | 0.55~0.75 의 ChCODH2 그룹 | 28/19/14 가 재현되는가. 대표가 ChCODH2 인 적 있는가 |
| 5 identity 세트 | 0.35~0.60 | 기준을 올리면 포개지는가 |
| **6 같은 잣대** | **두 방식 전 후보를 한 표에** | **n_effective 와 redundancy** ← 이 노트북의 결론 |
| 7 겹침 | 자카드 행렬 | 한쪽에만 있는 서열이 무엇인가 |
| 8 기준 민감도 | 0.05 움직였을 때 | 01 번의 0.40~0.55 불안정이 여기서도 보이나 |
| 9 그림 | 선택 (matplotlib 없으면 표) | 분포와 n vs n_effective |
| 10 저장·요약 | tsv + params.json | 선택지와 trade-off. **결론은 사용자가 낸다** |

## 작업 폴더

```
workspace/ipr010047_mmseqs/
├─ input/                                  공용 (2절이 원본에서 링크를 건다)
│     protein-matching-IPR010047.fasta
│     master_1827.tsv
│     chcodh2_alignment.tsv                ← 이번에 새로 링크된다
├─ script/
│  ├─ 20261025_01_cluster_sweep/           (01 번 묶음. 2026-10-25b 사본을 쓴다)
│  └─ 20261025_02_conservation_set/        ← 받은 2개를 여기 둔다
│        cluster_set_compare.ipynb
│        chcodh2_sweep.py                  이름을 바꾸지 말 것 (2026-10-25c)
├─ result/
│  ├─ clusters/c0.8_cm1_s7.5/              01 번이 만든 것. **다시 돌리지 않는다**
│  │     id10_cluster.tsv … id90_cluster.tsv
│  ├─ 20261025_01_cluster_sweep/           01 번 결과
│  └─ 20261025_02_conservation_set/        ← 이 묶음의 표
└─ tmp/
```

```bash
RUN=20261025_02_conservation_set
mkdir -p /mnt/af2results/mingyu/workspace/ipr010047_mmseqs/script/$RUN
# 다운로드한 cluster_set_compare.ipynb · chcodh2_sweep.py 를 그 폴더로 옮긴다
```

**이 노트북은 mmseqs 를 실행하지 않는다.** 이미 있는 산출물만 읽으므로 몇 초 만에 끝난다.
의존성은 pandas + numpy 뿐이고 matplotlib 은 있으면 쓴다(9절).

## 확인이 필요한 것 (2절이 자동으로 띄운다)

- ⚠ `chcodh2_alignment.tsv` 에 **헤더가 있는지 미확인.** 열 순서는 복원된 `--format-output`
  (`query,target,fident,alnlen,qcov,tcov,evalue,bits,qstart,qend,tstart,tend,qaln,taln`) 으로
  추정한다. 노트북이 헤더 유무를 판정하고 **무엇을 근거로 열 이름을 정했는지 찍는다.**
  틀렸으면 1절의 `ALIGN_COLUMNS` 에 순서를 직접 적으면 된다.
- ⚠ `master_1827.tsv` 의 `cf`·`cband`·`r41…`·D-cluster 유형 **컬럼명이 미확인.** 있으면 쓰고
  없으면 건너뛰되, **건너뛴 사실을 2절이 출력한다.**
- ⚠ `r612` 가 논문의 **V610** 과 2 어긋난다. 질의가 `pdb|1SU6|A` 였으므로 PDB 번호가 섞였을
  수 있다. **세트 선택에는 영향이 없지만 보존도 계산 전에 번호 체계를 정해야 한다.**

## 1. 설정

경로와 상수를 전부 여기 모았다. 환경이 바뀌면 이 두 셀만 고치고 1절부터 다시 실행한다.
**셀이 두 개인 이유** — 앞은 경로와 모듈(01 번 노트북과 같은 꼴), 뒤는 이번 분석의
후보 목록과 기준값이다. 후보를 바꿔 다시 돌릴 때는 뒤쪽 셀만 고치면 된다.

`CLUSTERS` 는 **01 번 묶음이 만든 폴더**다. 이 노트북은 그 안의 `id*_cluster.tsv` 를
읽기만 하고 mmseqs 를 돌리지 않는다.

In [ ]:
import json, sys
from pathlib import Path
import pandas as pd

# ── 고칠 곳은 여기뿐이다 ────────────────────────────────────────────
BASE   = Path("/mnt/af2results/mingyu/workspace/ipr010047_mmseqs")
RUN    = "20261025_02_conservation_set"   # 이 묶음의 이름. script/ 와 result/ 가 이걸로 묶인다
NEEDS  = "2026-10-25c"                    # 같이 받은 chcodh2_sweep.py 의 버전. 다르면 멈춘다.

SCRIPT = BASE / "script" / RUN            # 이 노트북과 chcodh2_sweep.py 가 같이 있는 곳
sys.path.insert(0, str(SCRIPT))
try:
    import chcodh2_sweep as cs
except ModuleNotFoundError:
    raise SystemExit(f"{SCRIPT}/chcodh2_sweep.py 가 없다 — 노트북과 같은 폴더에 두고 다시 실행할 것.")
if cs.VERSION != NEEDS:
    raise SystemExit(f"버전 불일치: 노트북 {NEEDS} vs 모듈 {cs.VERSION}. 둘 다 최신으로 받을 것.")

PARAMS = f"c{cs.FIXED_COV}_cm{cs.FIXED_COV_MODE}_s{cs.FIXED_SENSITIVITY}"
INPUT, OUTDIR = BASE / "input", BASE / "result" / RUN    # 입력은 공용, 결과는 묶음별
CLUSTERS = BASE / "result" / "clusters" / PARAMS   # 01 번 산출물. 읽기만 한다 (mmseqs 미실행)
FASTA, MASTER = INPUT / "protein-matching-IPR010047.fasta", INPUT / "master_1827.tsv"
ALIGN = INPUT / "chcodh2_alignment.tsv"
for d in (INPUT, OUTDIR): d.mkdir(parents=True, exist_ok=True)
print(f"묶음 {RUN}\n모듈 {cs.__file__} (v{cs.VERSION})\n클러스터 {CLUSTERS}\n결과 {OUTDIR}")

이쪽이 **이번 분석의 손잡이**다.

- `CLUSTER_T` / `ID_CUTOFFS` — 비교할 후보. 01 번에서 ChCODH2 그룹이 0.60→28, 0.65→19,
  0.70→14 였으므로 그 주변을 잡았다. identity 쪽은 중앙값 0.379 와 95% 지점 0.55 사이를 쓸었다.
- `FINE_T = 0.90` — `n_effective` 의 분모가 되는 미세 분할. 01 번에서 0.90 은 921 클러스터이고
  NMI 가 0.90 이상이라 상위 분할에 깔끔히 포개진다. 즉 "거의 같은 서열" 을 세는 자로 쓸 수 있다.
- `ALIGN_COLUMNS` — `None` 이면 노트북이 헤더 유무를 판정해 스스로 정한다. 2절 출력이
  이상하면 `cs.ALIGN_FORMAT_OUTPUT` (복원된 `--format-output` 순서) 를 넣는다.
- `TARGET_EFFECTIVE` — 10절 요약에서 **두 방식에 같은 분모를 물려** 비교할 기준.
  이 값을 바꾸면 요약의 후보가 바뀐다. 결론을 정하는 값이 아니라 **질문을 정하는 값**이다.

In [ ]:
REF_ACC    = cs.CHCODH2_ACC                     # Q9F8A8 — 설계 대상. 빠지면 세트가 무의미하다
FINE_T     = 0.90                               # n_effective 의 분모 (중복 걷어내는 자)
CLUSTER_T  = [0.55, 0.60, 0.65, 0.70, 0.75]     # A. 클러스터 기반 후보
ID_CUTOFFS = [0.35, 0.40, 0.45, 0.50, 0.55, 0.60]   # B. identity 기반 후보
STAB_T     = [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75]  # 8절 민감도 (0.40~0.55 전부)
TARGET_EFFECTIVE = 12            # 10절에서 두 방식에 물릴 같은 분모. 바꿔도 된다
STABLE_JACCARD   = 0.90          # 인접 기준 간 자카드가 이 값 이상이면 '안정'
ALIGN_COLUMNS    = None          # 열 이름이 안 맞으면 cs.ALIGN_FORMAT_OUTPUT 를 넣을 것

# 01 번 묶음에서 확인된 값. 2·3·4절이 자동 대조한다. 어긋나면 입력이 달라진 것이다.
KNOWN_REF_CLUSTER   = {0.60: 28, 0.65: 19, 0.70: 14, 0.80: 4, 0.90: 4}
KNOWN_FIDENT_MEDIAN = 0.379      # ChCODH2 대비 identity 중앙값 (95% 가 0.55 이하)
KNOWN_N_SEQ         = 1827

# 입력 원본의 현재 위치. 2절이 input/ 에 없으면 여기서 링크를 건다 (원본은 읽기만 한다).
DOCK = Path("/mnt/af2results/mingyu/workspace/docking_simulation/result")
SRC = {FASTA.name:  Path("/mnt/af2results/mingyu/chcodh2") / FASTA.name,
       MASTER.name: DOCK / "homolog_split" / MASTER.name,
       ALIGN.name:  DOCK / "homolog_selection" / ALIGN.name}
print(f"후보 A(클러스터) {CLUSTER_T}\n후보 B(identity) {ID_CUTOFFS}\n"
      f"미세 분할 {FINE_T}, 같은 분모 기준 n_effective>={TARGET_EFFECTIVE}")

## 2. 입력 확인 — 숫자를 눈으로 본다

`input/` 에 파일이 없으면 **이 셀이 원본에서 링크를 건다.** 터미널에서 따로 할 일은 없다.

그다음 세 가지를 센다. **여기서 숫자가 다르면 아래 전부가 01 번 결과와 비교 불가**이므로
멈춰야 한다.

1. **FASTA** — 1,827 레코드인가. ChCODH2(Q9F8A8) 가 들어 있는가 (없으면 예외).
2. **alignment** — 열 이름을 **무엇을 근거로 정했는지**, `fident` 가 분수인지 퍼센트인지,
   HSP 중복을 몇 건 병합했는지, 그리고 **1,827 을 다 덮는지**. 복원된 명령이
   `--max-seqs 100000 -e 10000` 이었으므로 절단은 없어야 한다. 모자라면 그 수와 예시를 찍는다.
3. **master** — `cf`·`cband`·`r41…`·D-cluster 유형 컬럼을 찾는다. **없으면 건너뛰되 그
   사실을 출력한다** (6절의 구성 열이 비는 이유가 된다).

마지막으로 **정렬 파일이 정말 ChCODH2 대비 `fident` 인지** 네 가지 불변식으로 못 박는다
(`cs.assert_alignment_usable`). 헤더도 `--columns` 도 없는 파일에서 열이 한 칸만 밀리면
`fident` 자리에 `qcov`(0.95) 나 `qstart`(1) 가 들어오는데 **둘 다 0~1 범위라 척도 판정을
그냥 통과한다.** 그러면 5절의 identity 세트가 전부 1,827 전체가 되고 6절 표는 멀쩡해 보인다 —
이 노트북에서 가장 나쁜 실패다. 그래서 ① 자기 히트 `fident` 는 1.0, ② 전체가 상수일 수 없음,
③ universe 밖 target 이 없음(있으면 절단이 아니라 **accession 표기 불일치**이고 조인이
조용히 빈다), ④ `fident>=0.999` 는 자기 히트 1건뿐 — 어긋나면 **경고가 아니라 예외**다.

마지막 줄이 **`r612` vs 논문 `V610` 불일치**다. 세트 선택에는 영향이 없지만,
보존도를 계산하기 전에 **어느 번호 체계로 통일할지 사람이 정해야 한다.**

In [ ]:
print("\n".join(cs.link_inputs([FASTA, MASTER, ALIGN], SRC)))

universe, uni = cs.fasta_universe(FASTA)
print(f"\nFASTA 레코드 {uni['n_records']} / 고유 accession {uni['n_unique_accession']} "
      f"(기대 {KNOWN_N_SEQ}),  중복 {uni['n_duplicate_accession']},  빈 칸 {uni['n_empty_accession']}")
if REF_ACC not in universe:
    raise SystemExit(f"{REF_ACC} 가 FASTA 에 없다 — 설계 대상이 빠진 세트는 의미가 없다.")

align, am = cs.read_alignment(ALIGN, columns=ALIGN_COLUMNS, ref_acc=REF_ACC)
cov = cs.alignment_coverage(align, universe)
print(f"\nalignment 열 이름 출처: {am['column_source']}  (헤더 판정: {am['header_reason']})"
      f"\n  fident 척도 {am['fident_scale']},  결측 {am['n_fident_missing']}건")
print(f"  행 {am['n_rows']} → target {am['n_unique_target']}개  "
      f"(HSP {am['n_merged']}건 병합, 규칙: {am['hsp_tiebreak']})")
print(f"  universe 피복 {cov['n_in_universe']}/{cov['n_universe']},  결측 {cov['n_missing']}"
      + (f" 예: {cov['missing_examples'][:5]} ← 절단 의심" if cov["n_missing"] else " ← 절단 없음"))
# 아래 네 줄이 '조용히 틀린 숫자' 를 막는 곳이다. 열이 한 칸 밀려 fident 자리에
# qcov(0.95)·qstart(1) 가 들어오거나 accession 표기가 달라 조인이 비면 여기서 예외가 난다.
for line in cs.assert_alignment_usable(align, am, cov, ref_acc=REF_ACC):
    print("  " + line)

### 2-2. master 의 부가 컬럼과 잔기 번호 체계

`master_1827.tsv` 에 `cf`·`cband`·`r41 r43 r57 r59 r583 r612`·D-cluster 유형(SF4/SF2)이
들어 있다고 보고됐지만 **실제 컬럼명이 미확인**이다. 후보 이름으로 찾아서 있으면 쓰고
없으면 건너뛰되, **건너뛴 목록을 출력한다** — 6절의 구성 열이 비는 이유가 그것이다.

아래쪽이 **이 노트북이 사용자에게 띄우는 결정 사항**이다. `r612` 가 논문의 **V610** 과 2
어긋난다. 질의가 `pdb|1SU6|A` 였으므로 PDB 번호와 UniProt 번호가 섞였을 수 있다.
**세트 선택(이 노트북)에는 영향이 없지만, 보존도를 계산하기 전에 어느 번호 체계로
통일할지 정해야 한다.** 노트북이 대신 고치지 않는다.

In [ ]:
master, mm = cs.read_master_table(MASTER)
if master is None:
    print(f"master 없음 ({mm['reason']}) — D-type·cband 구성은 전부 건너뛴다")
else:
    print(f"master {mm['n_rows']}행 / {mm['n_assignments']}배정\n  전체 컬럼 {mm['columns']}")
    print(f"  찾은 컬럼   {mm['found']}\n  없는 컬럼   {mm['missing']}  ← 이 열은 건너뛴다")

chk = cs.check_position_numbering(mm.get("position_columns", {}))
print("\n■ 잔기 번호 대조 (논문 번호 = " + str(list(cs.PAPER_POSITIONS)) + ")")
for line in chk["messages"]:
    print("  " + line)
print(f"\n사람이 정할 것 남음: {chk['needs_decision']}  "
      f"(세트 선택에는 영향 없음 — 6절 이후 숫자는 번호 체계와 무관하다)")

## 3. ChCODH2 대비 identity 분포

B 방식의 기준을 어디에 둘지 보려면 먼저 분포를 봐야 한다. 01 번에서 **중앙값 0.379,
95% 가 0.55 이하**로 관측됐다. 그 두 값과 대조해 입력이 같은 파일인지 확인한다.

히스토그램은 표로 찍는다(`bar` 열이 막대다). matplotlib 이 없는 환경에서도 분포 모양이
보이고, 9절 그림은 선택이다.

마지막 표가 **기준별 세트 크기**다. `frac_of_universe` 가 0.1 을 넘으면 1,827 중 10% 이상이
들어온다는 뜻이고, 그 구간은 이미 "근연 상동체" 가 아니라 패밀리 전체에 가깝다.

In [ ]:
fs = cs.fident_summary(align, universe=universe)
print(f"n={fs['n']}   최소 {fs['min']:.3f}   중앙 {fs['median']:.3f}   최대 {fs['max']:.3f}   "
      f"자기 자신(1.0) {fs['n_eq_one']}건")
print(f"q25 {fs['q25']:.3f}  q75 {fs['q75']:.3f}  q90 {fs['q90']:.3f}  q95 {fs['q95']:.3f}"
      f"   0.55 이하 비율 {fs['frac_le_0p55']:.3f}")

gap = abs(fs["median"] - KNOWN_FIDENT_MEDIAN)
print(f"\n01번 관측 대조 — 중앙값 {KNOWN_FIDENT_MEDIAN}: "
      + ("일치" if gap < 0.01 else f"⚠ {gap:.3f} 차이, 입력이 다를 수 있다"))
print(f"                 95% 지점 0.55 이하: "
      + ("맞다" if fs["q95"] <= 0.56 else f"⚠ 어긋난다 (q95 {fs['q95']:.3f})"))

hist = cs.fident_histogram(align, universe=universe)
print(f"\n■ 히스토그램 (구간 폭 0.05, n_at_or_above = 그 값 이상인 서열 수)")
print(hist.round(4).to_string(index=False))

print(f"\n■ 기준별 세트 크기 (ChCODH2 자신 포함)")
cs.identity_cutoff_table(align, ID_CUTOFFS + [0.65, 0.70], universe, ref_acc=REF_ACC).round(4)

## 4. A 방식 — 클러스터 기반 세트

`result/clusters/c0.8_cm1_s7.5/` 의 **기존 파일을 읽는다. mmseqs 를 돌리지 않는다.**
파일이 없으면 예외가 나므로, 1절의 `CLUSTERS` 가 01 번 묶음의 폴더를 가리키는지 확인할 것.

먼저 `FINE_T`(0.90) 분할을 읽는다. 이것이 `n_effective` 의 분모이므로 **universe 를 다 덮는지
반드시 확인한다** — 덮지 못하면 중복 제거가 조용히 틀어지므로 거기서 멈춘다.

그다음 임계값마다 ChCODH2 그룹의 크기와 대표를 찍는다. 01 번 값(0.60→28, 0.65→19, 0.70→14)이
재현되면 `= 01번` 이 붙는다. 마지막 줄이 **`ref_is_mm_rep` 문제**다 — ChCODH2 가 한 번도
대표가 아니면, 대표 서열만 뽑아 쓰는 방식에서는 설계 대상이 빠진다.

In [ ]:
fine, fmeta = cs.load_existing_partition(CLUSTERS, FINE_T)
print(f"미세 분할 {fmeta['label']}: {fmeta['n_assignments']}배정 / {fmeta['n_clusters']}클러스터 "
      f"(01번 0.90 → 921)")
not_covered = [a for a in universe if a not in fine.index]
if not_covered:
    raise SystemExit(f"{fmeta['label']} 가 universe 를 못 덮는다 ({len(not_covered)}개 누락, "
                     f"예 {not_covered[:5]}) — n_effective 의 분모가 틀어지므로 멈춘다.")

cl_part, cl_sets = {}, {}
for t in CLUSTER_T:
    ser, meta = cs.load_existing_partition(CLUSTERS, t)
    cl_part[t], cl_sets[t] = ser, cs.cluster_set(ser, REF_ACC)
    rep = str(ser.loc[REF_ACC])
    known = KNOWN_REF_CLUSTER.get(round(t, 2))
    chk = "" if known is None else ("  = 01번" if len(cl_sets[t]) == known else f"  ⚠ 01번은 {known}")
    print(f"min-seq-id {t:.2f}   전체 {ser.nunique():5d}클러스터   ChCODH2 그룹 {len(cl_sets[t]):4d}개"
          f"   대표 {rep}{' (=ChCODH2)' if rep == REF_ACC else ''}{chk}")

is_rep = [t for t in CLUSTER_T if str(cl_part[t].loc[REF_ACC]) == REF_ACC]
print(f"\nChCODH2 가 클러스터 대표인 임계값: {is_rep if is_rep else '없음'}"
      + ("" if is_rep else "  ← 01번과 같다. 대표 서열만 쓰는 방식이면 설계 대상이 빠진다."))

## 5. B 방식 — identity 기반 세트

`fident >= 기준` 인 서열 전부. 클러스터와 달리 **멤버끼리는 서로 멀어도 된다** — ChCODH2 를
중심으로 한 방사형 세트다.

`include_reference=True` 로 **ChCODH2 자신을 항상 넣는다.** 자기 히트의 fident 는 1.0 이라
어차피 통과하지만, 세트의 정의를 기준값에 의존하게 두지 않는 쪽이 안전하다(4절에서 본
`ref_is_mm_rep` 문제와 같은 종류의 사고를 막는다).

마지막 줄의 **중첩(nested)** 확인이 중요하다. identity 기반은 정의상 기준을 올리면 세트가
작아지며 포개져야 한다. `False` 가 나오면 fident 에 결측이 섞였거나 열이 어긋난 것이다.

마지막 줄은 **이 비교가 성립하는지** 묻는다. 가장 느슨한 기준이 1,827 전체를 삼키거나 가장 엄한 기준과 같은 세트를 내놓으면 identity 축에 정보가 없다는 뜻이므로(열 밀림 또는 후보 범위가 분포 밖) **예외로 멈춘다.**

In [ ]:
id_sets = {c: cs.identity_set(align, c, universe, ref_acc=REF_ACC, include_reference=True)
           for c in ID_CUTOFFS}
for c in ID_CUTOFFS:
    s = id_sets[c]
    n_no_aln = sum(1 for a in s if a not in align.index)
    print(f"fident >= {c:.2f}   {len(s):5d}개   ChCODH2 포함 {REF_ACC in s}"
          f"   정렬 결과 없는 멤버 {n_no_aln}개"
          + ("  ← include_reference 로 넣은 것" if n_no_aln else ""))

nested = all(id_sets[hi] <= id_sets[lo] for lo, hi in zip(ID_CUTOFFS[:-1], ID_CUTOFFS[1:]))
print(f"\n기준을 올리면 세트가 포개지는가(중첩): {nested}")
if not nested:
    raise SystemExit("identity 기반 세트가 중첩이 아니다 — fident 결측이나 열 어긋남을 의심할 것.")
loose, tight = id_sets[ID_CUTOFFS[0]], id_sets[ID_CUTOFFS[-1]]
print(f"가장 느슨한 {ID_CUTOFFS[0]:.2f} 세트가 universe 의 {len(loose) / len(universe):.1%} 를 "
      f"차지하고, 가장 엄한 {ID_CUTOFFS[-1]:.2f} 와 {len(loose) - len(tight)}개 차이 난다.")
if len(loose) == len(universe) or loose == tight:
    raise SystemExit(                      # identity 축이 평평하면 6~8절 비교가 성립하지 않는다
        f"identity 축이 정보를 담고 있지 않다 (느슨 {len(loose)} / 엄함 {len(tight)} / "
        f"universe {len(universe)}). fident 열이 밀렸거나 후보 범위가 분포 밖이다 — "
        f"3절 히스토그램을 보고 ID_CUTOFFS 를 다시 잡거나 ALIGN_COLUMNS 를 줄 것.")

## 6. 같은 잣대로 비교 ★이 노트북의 핵심★

두 방식의 **모든 후보를 한 표에** 올린다. 같은 함수(`set_profile`)로 재므로 숫자의 뜻이 같다.

| 열 | 뜻 | 어떻게 읽나 |
|---|---|---|
| `n` | 서열 수 | **이걸로 줄 세우지 않는다.** 거의 같은 서열 28개는 14개보다 나을 게 없다 |
| `n_effective` | 0.90 분할 기준 서로 다른 클러스터 수 | **보존도의 실제 분모.** 표는 이것으로 정렬된다 |
| `redundancy` | `1 - n_effective/n` | 0.6 이면 세트의 60% 가 중복이다 |
| `has_reference` | ChCODH2 포함 여부 | `False` 면 그 후보는 쓸 수 없다 |
| `fident_min/median/max` | 세트 내 ChCODH2 대비 identity | `fident_min` 이 낮으면 먼 서열을 끌어온 것 |
| `n_dtype_sf4/sf2` | D-cluster 유형 구성 | master 에 컬럼이 없으면 열이 빠진다(2절 참고) |

같은 `n_effective` 를 **더 적은 서열로, 더 높은 fident 하한으로** 얻는 쪽이 유리하다.
다만 그것이 곧 "더 좋은 세트" 인지는 생물학적 판단이므로 여기서 단정하지 않는다.

In [ ]:
prof = [cs.set_profile(cl_sets[t], universe=universe, align_df=align, fine_partition=fine,
                       master=master, ref_acc=REF_ACC, method="A.클러스터",
                       name=f"cluster@{t:.2f}", cutoff=t) for t in CLUSTER_T]
prof += [cs.set_profile(id_sets[c], universe=universe, align_df=align, fine_partition=fine,
                        master=master, ref_acc=REF_ACC, method="B.identity",
                        name=f"fident>={c:.2f}", cutoff=c) for c in ID_CUTOFFS]

table = cs.profile_table(prof)          # n 이 아니라 n_effective 로 줄 세운다
print(f"후보 {len(table)}개 = 클러스터 {len(CLUSTER_T)} + identity {len(ID_CUTOFFS)}")
print(f"ChCODH2 가 빠진 후보: {table.loc[~table['has_reference'], 'name'].tolist() or '없음'}")
print(f"fident 결측 멤버가 있는 후보: "
      f"{table.loc[table['n_fident_missing'].fillna(0) > 0, 'name'].tolist() or '없음'}")

shown = ["name", "method", "n", "n_effective", "redundancy", "has_reference",
         "fident_min", "fident_median", "fident_max"]
extra = [c for c in ("n_dtype_sf4", "n_dtype_sf2", "n_dtype_other", "n_dtype_missing",
                     "cband_top") if table[c].notna().any()]
if not extra:
    print("※ master 에 D-type·cband 컬럼이 없어 구성 열은 비운다 (2절 '없는 컬럼' 참고)")
table[shown + extra].round(3)

## 7. 두 방식이 얼마나 겹치나

자카드 행렬이다. 1.0 이면 같은 세트, 0 이면 완전히 다르다. 어떤 조합이 서로 거의 같다면
**그 둘 사이에서는 어느 방식을 고르든 결과가 같다**는 뜻이므로 선택이 쉬워진다.

그다음이 더 중요하다 — **한쪽에만 있는 서열이 무엇인가.** identity 기반에만 있는 서열은
"ChCODH2 와는 가깝지만 클러스터에는 안 묶인" 것이고(정렬 피복이 낮거나 단편일 수 있다),
클러스터에만 있는 서열은 "ChCODH2 와는 멀지만 같은 덩어리인" 것이다. 예시의 `fident` 를
같이 찍으므로 어느 쪽인지 바로 보인다.

In [ ]:
A = {f"cl@{t:.2f}": cl_sets[t] for t in CLUSTER_T}
B = {f"id>={c:.2f}": id_sets[c] for c in ID_CUTOFFS}
jac = cs.overlap_matrix(A, B, value="jaccard")
print("■ 자카드 (행 = A.클러스터 기반, 열 = B.identity 기반)")
print(jac.round(3).to_string())

pairs = sorted(((a, b, float(jac.loc[a, b])) for a in jac.index for b in jac.columns),
               key=lambda r: -r[2])
print("\n■ 가장 비슷한 조합 3개")
for a, b, j in pairs[:3]:
    print(f"   {a} vs {b}   자카드 {j:.3f}")

PAIR_A, PAIR_B, _ = pairs[0]
OV = cs.set_overlap(A[PAIR_A], B[PAIR_B], max_examples=8)
print(f"\n■ {PAIR_A} vs {PAIR_B}:  공통 {OV['n_intersection']},  "
      f"{PAIR_A} 에만 {OV['n_only_a']},  {PAIR_B} 에만 {OV['n_only_b']}")
for lab, accs in ((f"{PAIR_A} 에만", OV["only_a_examples"]),
                  (f"{PAIR_B} 에만", OV["only_b_examples"])):
    if accs:
        print(f"   {lab} (예시 {len(accs)}개): " + ", ".join(
            f"{x} fident={align['fident'].get(x, float('nan')):.3f}" for x in accs))

## 8. 기준 민감도 — 0.05 움직이면 세트가 얼마나 바뀌나

**01 번에서 ChCODH2 의 클러스터 소속이 0.40~0.55 에서 불안정했다.** 대표가 바뀌며 그룹을
건너뛰어 0.45→36개, 0.50→104개, 0.55→6개로 튀었다. 그 구간을 기준으로 삼으면 임계값을
조금만 달리 적어도 전혀 다른 세트가 나온다는 뜻이다.

클러스터 쪽 민감도는 그래서 **0.40 부터** 쓸어 그 구간 전체를 덮는다.

여기서는 두 방식 모두에 같은 질문을 던진다 — **인접한 기준 사이의 자카드.** 1.0 에 가까우면
기준을 조금 움직여도 세트가 같고(안정), 낮으면 그 구간이 경계다.

identity 기반은 정의상 중첩이므로 `n_only_high` 는 0 이어야 한다. 즉 identity 쪽의 불안정은
"멤버가 바뀌는" 것이 아니라 **"몇 개가 한꺼번에 빠지는"** 형태로만 나타난다. 클러스터 쪽은
양쪽 모두 0 이 아닐 수 있다 — 그것이 01 번에서 본 **그룹 건너뛰기**다.

In [ ]:
stab_cl = cs.cutoff_stability(
    lambda t: cs.cluster_set(cs.load_existing_partition(CLUSTERS, t)[0], REF_ACC),
    STAB_T, ref_acc=REF_ACC, stable_jaccard=STABLE_JACCARD, label="A.클러스터")
stab_id = cs.cutoff_stability(
    lambda c: cs.identity_set(align, c, universe, ref_acc=REF_ACC),
    ID_CUTOFFS + [0.65, 0.70], ref_acc=REF_ACC, stable_jaccard=STABLE_JACCARD,
    label="B.identity")
stab = pd.concat([stab_cl, stab_id], ignore_index=True)
print(stab.round(3).to_string(index=False))

print(f"\n(자카드 >= {STABLE_JACCARD} 를 '안정' 으로 본다)")
for lab, g in stab.groupby("label"):
    w = g.sort_values("jaccard", kind="mergesort").iloc[0]
    swap = g[(g["n_only_low"] > 0) & (g["n_only_high"] > 0)]
    print(f"  {lab}: 안정 {int(g['is_stable'].sum())}/{len(g)} 구간,  "
          f"가장 크게 바뀌는 곳 {w['cutoff_low']:.2f}→{w['cutoff_high']:.2f} "
          f"자카드 {w['jaccard']:.3f} (n {int(w['n_low'])}→{int(w['n_high'])})")
    print(f"      멤버가 서로 교체되는(그룹 건너뛰기) 구간: "
          + (", ".join(f"{r.cutoff_low:.2f}→{r.cutoff_high:.2f}"
                       for r in swap.itertuples()) if len(swap) else "없음 — 포개지기만 한다"))

## 9. 그림 (선택)

matplotlib 이 있으면 두 장. 왼쪽은 identity 분포(빨간 점선이 01 번 중앙값 0.379, 회색이 각
후보 기준), 오른쪽이 **`n` 대 `n_effective`** 다. 회색 대각선이 `n_effective = n`
(중복이 전혀 없는 이상적인 경우)이고, 점이 그 선에서 아래로 멀어질수록 중복이 많다.

**오른쪽 그림이 6절 표의 요점이다** — 같은 높이(같은 분모)에서 왼쪽에 있는 점이 더 적은
서열로 같은 다양성을 얻은 것이다. 두 방식의 세트 크기가 10배 넘게 차이 나므로 **양축 모두
로그**다. 로그축에서도 회색 대각선은 직선이고, 점이 선에서 아래로 떨어진 거리가 중복이다.

**축 라벨은 영문이다** — matplotlib 에 한글 폰트가 없는 환경에서 한글이 □ 로 깨지기 때문이다.
matplotlib 이 없으면 안내만 찍고 넘어간다. 3·6절 표로 판단은 충분히 된다.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib 이 없다. 3·6절 표로 충분하다. (설치하지 않고 넘어가도 된다)")

if plt is not None:                                  # 축 이름은 한글 글꼴 경고를 피해 영문
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].bar(hist["bin_low"] + 0.025, hist["n"], width=0.045)
    for c in ID_CUTOFFS:
        ax[0].axvline(c, ls=":", c="gray", lw=0.8)
    ax[0].axvline(KNOWN_FIDENT_MEDIAN, ls="--", c="red")
    ax[0].set_xlabel("fident vs ChCODH2"); ax[0].set_ylabel("n sequences")
    ax[0].set_title("identity distribution (red = 2026-10 median)")

    names = {"A.클러스터": "A cluster-based", "B.identity": "B identity-based"}
    for m, mark in (("A.클러스터", "o-"), ("B.identity", "s-")):
        sub = table[table["method"] == m].sort_values("n")
        ax[1].plot(sub["n"], sub["n_effective"], mark, label=names[m])
    top = float(table["n"].max())
    ax[1].plot([1, top], [1, top], ls="--", c="gray", lw=0.8)
    ax[1].set_xscale("log"); ax[1].set_yscale("log")   # 세트 크기가 10배씩 차이 난다
    ax[1].set_xlabel("n (set size)"); ax[1].set_ylabel("n_effective (distinct id90 clusters)")
    ax[1].set_title("gray line = no redundancy"); ax[1].legend(loc="upper left")
    fig.tight_layout(); plt.show()

## 10. 저장과 요약

표를 tsv 로, 입력 점검 결과를 json 으로, 재현 정보(입력 해시·버전·후보 목록)를
`params.json` 으로 남긴다. `set_members.tsv` 에는 **각 후보의 멤버 전부**가 들어가므로,
다음 단계(보존도 계산)는 이 파일 하나만 읽으면 된다.

마지막이 사람이 읽는 요약이다. **결론은 비워 둔다.** 각 방식의 범위와,
`TARGET_EFFECTIVE`(=`n_effective` 목표)를 같이 물렸을 때 두 방식이 각각 몇 서열로
그것을 채우는지, 그리고 무엇을 잃는지만 적는다.

**다음에 정할 것 세 가지**
1. 세트 — 이 노트북의 표를 보고 하나 고른다 (A 의 어느 임계값인지, B 의 어느 기준인지).
2. 번호 체계 — 2절의 `r612` vs `V610`. 보존도 계산 전에 통일해야 한다.
3. 정렬 방법 — 세트가 정해지면 그 서열들로 MSA 를 다시 뜰지, 기존 `qaln/taln`
   (`03_equiv_map.py` 의 등가 매핑)을 쓸지.

In [ ]:
table.to_csv(OUTDIR / "set_profiles.tsv", sep="\t", index=False)
jac.to_csv(OUTDIR / "set_overlap_jaccard.tsv", sep="\t")
stab.to_csv(OUTDIR / "set_cutoff_stability.tsv", sep="\t", index=False)
hist.to_csv(OUTDIR / "fident_histogram.tsv", sep="\t", index=False)
pd.DataFrame([dict(name=p["name"], method=p["method"], cutoff=p["cutoff"], n=p["n"],
                   n_effective=p["n_effective"], members=";".join(p["members"]))
              for p in prof]).to_csv(OUTDIR / "set_members.tsv", sep="\t", index=False)
(OUTDIR / "inputs_check.json").write_text(json.dumps(
    dict(alignment=am, coverage=cov, fident=fs, master=mm,
         position_check=cs.check_position_numbering(mm.get("position_columns", {})),
         best_pair=dict(cluster=PAIR_A, identity=PAIR_B, overlap=OV)),
    indent=2, ensure_ascii=False, default=str))
cs.write_params(OUTDIR, "보존도 세트 비교 (notebook)", inputs=[FASTA, MASTER, ALIGN],
                outputs=sorted(OUTDIR.glob("*.tsv")), commands=["(mmseqs 미실행 — 기존 산출물만 읽음)"],
                cluster_thresholds=CLUSTER_T, identity_cutoffs=ID_CUTOFFS, fine_threshold=FINE_T,
                target_effective=TARGET_EFFECTIVE, stable_jaccard=STABLE_JACCARD,
                align_column_source=am["column_source"], fident_scale=am["fident_scale"],
                master_missing_columns=mm.get("missing"), clusters_dir=str(CLUSTERS))
print(f"저장 위치 {OUTDIR}")
for f in sorted(OUTDIR.glob("*.tsv")) + sorted(OUTDIR.glob("*.json")):
    print(f"   {f.name:28s} {f.stat().st_size:>10,} bytes")
print("\n■ 요약 — 선택지와 trade-off. 판단은 사용자가 한다")
for line in cs.compare_summary(table, target_effective=TARGET_EFFECTIVE,
                               overlap=OV, stability=stab):
    print("  • " + line)